In [13]:
import torch

In [14]:
import tarfile

import requests


def download_dataset():
    images_url = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz"
    labels_url = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat"
    download_and_extract(images_url, "data/102flowers.tgz")
    download_file(labels_url, "data/imagelabels.mat")

def download_and_extract(url, output_path):
    response = requests.get(url)
    with open(output_path, "wb") as f:
        f.write(response.content)
    if output_path.endswith(".tgz"):
        with tarfile.open(output_path, "r:gz") as tar:
            tar.extractall(path="data")

def download_file(url, output_path):
    response = requests.get(url)
    with open(output_path, "wb") as f:
        f.write(response.content)
     


download_dataset()


In [22]:
from torchvision import transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [23]:
# Dataset class
import os
import scipy.io
from torch.utils.data import Dataset


class FlowersDataset(Dataset):
    def __init__(self,root_dir, transform=None):
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, "jpg")
        labels_mat = scipy.io.loadmat(os.path.join(root_dir, "imagelabels.mat"))
        self.labels = labels_mat["labels"][0]-1  # Convert to 0-based indexing
        self.transform = transform

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"image_{idx+1:05d}.jpg")
        image = self.load_image(img_path)
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label
    
    def load_image(self, img_path):
        from PIL import Image
        return Image.open(img_path).convert("RGB")
    


In [24]:
# Load dataset
dataset = FlowersDataset(root_dir="data", transform=transform)
print(f"Dataset size: {len(dataset)}")
# Example usage
image, label = dataset[0]
print(f"Image shape: {image.shape}, Label: {label}")


Dataset size: 8189
Image shape: torch.Size([3, 224, 224]), Label: 76


In [25]:
# Train val and test split
from torch.utils.data import random_split
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")



Train size: 5732, Val size: 1228, Test size: 1229


In [26]:
# dataloader on train and test set
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


Train batches: 180, Val batches: 39, Test batches: 39
